# Route C — Ensemble deployment (no grid search)

Streamlined train→test→**package** for the winning model: log-averaged
`bertweet-base` + `twitter-roberta-base-sentiment-latest` ensemble with
per-class threshold offsets. Final TEST macro-F1 = **0.7580**.

**This notebook does NOT re-run the grid search.** The winning config is known:
`lr=2e-5, weighted=True` (full inverse-frequency reweighting). It is hardcoded.

**Guarantee policy.** For each member the notebook first tries to *load the best
checkpoint your original run already saved* under `route_c_out/final_*`. If found,
no retraining happens and the packaged model is weight-for-weight the 0.7580 model.
If a checkpoint is missing it retrains that member with the fixed config (result
will be ~0.758, not bit-exact, because GPU/FP16 fine-tuning is nondeterministic).

Output: **`route_c_ensemble.model`** + a round-trip test that re-scores the pickle
on the test set so you can confirm the number before uploading to Hugging Face.

In [1]:
# ============================ CONFIG ============================
from pathlib import Path
import random, numpy as np

SPLIT_CSV   = "/kaggle/input/datasets/tanhvnh/sentiment-routec/data_split.csv"           # canonical split shared with Routes A/B
TEXT_COL    = None                        # None -> auto-detect
LABEL_COL   = None
SPLIT_COL   = None
TRAIN_KEYS  = {"train", "training"}
VAL_KEYS    = {"val", "valid", "validation", "dev"}
TEST_KEYS   = {"test", "testing", "holdout"}

# --- ensemble members (order matters only for printing) ---
MODELS = [
    "vinai/bertweet-base",
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
]
MAX_LEN    = 128
NUM_LABELS = 3

# --- WINNING CONFIG (no grid search) ---
BEST_LR       = 2e-5
BEST_WEIGHTED = True            # full inverse-frequency class weights
EPOCHS        = 5

# --- training (only used if a checkpoint must be retrained) ---
SEED          = 42
TRAIN_BS      = 16
EVAL_BS       = 32
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.06
EARLY_STOP    = 2
FREEZE_BOTTOM = 0
FP16            = True
GROUP_BY_LENGTH = True

# --- threshold tuning ---
THRESH_GRID   = None            # None -> np.linspace(-3, 3, 61) inside the tuner
# Set to [0.1, 0.2, 0.0] to FORCE the exact offsets from your 0.7580 run
# (bit-exact); leave None to recompute them from the loaded val logits.
FORCE_OFFSETS = None

# --- output / packaging ---
OUT_DIR    = Path("route_c_out"); OUT_DIR.mkdir(exist_ok=True)
MODEL_FILE = "route_c_ensemble.model"
INFER_BS   = 64                 # batch size baked into the pickled classifier

random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
print("Config loaded. Winning config: lr=%.0e, weighted=%s, epochs=%d"
      % (BEST_LR, BEST_WEIGHTED, EPOCHS))

Config loaded. Winning config: lr=2e-05, weighted=True, epochs=5


## 1. Deployment module + helpers

`sentiment_deploy_ensemble.py` must sit next to this notebook. Importing
`normalize_text` / `cardiff_preprocess` from it guarantees train-time and
serve-time cleaning are *literally the same code*.

In [2]:
%%writefile sentiment_deploy_ensemble.py
"""
sentiment_deploy_ensemble.py
============================

Self-contained, picklable deployment wrapper for the Route C ENSEMBLE
sentiment classifier (log-averaged BERTweet + twitter-roberta with per-class
threshold offsets), compatible with the case-manual API template.

The API loads a single ``*.model`` pickle and expects a dict::

    {"vectorizer": <obj with .transform(list[str])>,
     "classifier": <obj with .predict(X)>}

A two-model HuggingFace ensemble does not fit that interface, so this module
provides two adapters that replicate the notebook's scoring path EXACTLY:

* ``EnsembleVectorizer`` -- pass-through "vectorizer". Applies the same light
  cleaning used at training time (HTML strip + whitespace collapse) and returns
  the cleaned strings. ``fit``/``fit_transform`` are no-ops, so the wrapper is
  safe even if the API template calls ``fit_transform`` at inference time.

* ``EnsembleClassifier`` -- holds the fine-tuned weights, config and tokenizer
  files of BOTH members *inside the pickle* (no external paths), plus the
  per-class additive offsets fit on validation. At inference it reproduces the
  notebook's final decision rule:

      pred = argmax( mean_m logsoftmax(logits_m) + offsets )

  and maps internal class indices {0,1,2} -> API labels {-1, 0, 1}.

Each member is fed text in its OWN native form (BERTweet: raw cleaned, its
tokenizer normalises @USER/HTTPURL internally; twitter-roberta: mentions->@user,
links->http) so serve-time inputs match training-time inputs per model.

IMPORTANT (pickle/__main__ caveat): the API loads the pickle in a SEPARATE
process, so the classes referenced by the pickle must be importable there.
Defining them in THIS module (not in a notebook's __main__) is what makes the
round-trip work. Ship ``sentiment_deploy_ensemble.py`` alongside ``app.py``.
"""

from __future__ import annotations

import os
import tempfile
from typing import List, Sequence

# Internal index -> API label. Training uses 0=Negative, 1=Neutral, 2=Positive.
# The API/case-manual label space is -1=Negative, 0=Neutral, 1=Positive.
INDEX_TO_API_LABEL = {0: -1, 1: 0, 2: 1}


# --------------------------------------------------------------------------- #
# Text preprocessing (must match the training notebook bit-for-bit)
# --------------------------------------------------------------------------- #
def normalize_text(x) -> str:
    """Light, rule-based cleaning applied identically at train and serve time.

    Strips HTML (reviews contain markup) and collapses whitespace. Mention/URL
    normalisation is delegated to each model's own preprocessing so train and
    serve stay consistent.
    """
    if x is None:
        return ""
    x = str(x)
    if "<" in x and ">" in x:  # only pay BeautifulSoup cost when markup is likely
        try:
            from bs4 import BeautifulSoup
            x = BeautifulSoup(x, "html.parser").get_text(separator=" ")
        except Exception:
            pass
    x = " ".join(x.split())  # collapse all whitespace runs
    return x


def cardiff_preprocess(text) -> str:
    """twitter-roberta was trained with mentions -> '@user' and links -> 'http'.

    Applied ONLY to the roberta member (BERTweet's tokenizer does its own
    @USER/HTTPURL normalisation), matching the notebook's ``_prep_texts``.
    """
    out = []
    for tok in str(text).split(" "):
        if tok.startswith("@") and len(tok) > 1:
            tok = "@user"
        elif tok.startswith("http"):
            tok = "http"
        out.append(tok)
    return " ".join(out)


def _prep_for_member(texts: Sequence[str], is_bertweet: bool) -> List[str]:
    if is_bertweet:
        return list(texts)                      # tokenizer normalises internally
    return [cardiff_preprocess(t) for t in texts]


# --------------------------------------------------------------------------- #
# Vectorizer (pass-through, for API compatibility)
# --------------------------------------------------------------------------- #
class EnsembleVectorizer:
    """Pass-through 'vectorizer'. Tokenisation happens inside the classifier."""

    def fit(self, X=None, y=None):
        return self

    def transform(self, X: Sequence[str]) -> List[str]:
        if isinstance(X, str):
            X = [X]
        return [normalize_text(t) for t in X]

    def fit_transform(self, X: Sequence[str], y=None) -> List[str]:
        return self.transform(X)


# --------------------------------------------------------------------------- #
# Classifier (holds BOTH members + offsets; rebuilds lazily; never pickles
# live torch objects)
# --------------------------------------------------------------------------- #
class EnsembleClassifier:
    """Self-contained, picklable log-averaged ensemble classifier.

    Parameters
    ----------
    members : list of dicts, each::
        {"name": str,
         "is_bertweet": bool,
         "tokenizer_kwargs": dict,     # e.g. {"normalization": True, "use_fast": False}
         "config": transformers config,
         "state_dict": dict[str, cpu tensor],
         "tokenizer_files": dict[str, bytes]}
    offsets : sequence of 3 floats
        Per-class additive offsets (fit on validation in the notebook),
        applied to the averaged log-probabilities before argmax.
    max_length : int
    batch_size : int
    """

    def __init__(self, members=None, offsets=(0.0, 0.0, 0.0),
                 max_length: int = 128, batch_size: int = 64):
        self.max_length = int(max_length)
        self.batch_size = int(batch_size)
        self.index_to_api = dict(INDEX_TO_API_LABEL)
        self.offsets = [float(o) for o in offsets]
        # Picklable member payloads (already CPU tensors / raw bytes).
        self._members = members if members is not None else []
        # Live objects rebuilt lazily; never pickled.
        self._built = None

    # ---- (de)serialisation ------------------------------------------------ #
    def __getstate__(self):
        return {
            "max_length": self.max_length,
            "batch_size": self.batch_size,
            "index_to_api": self.index_to_api,
            "offsets": self.offsets,
            "_members": self._members,
        }

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._built = None

    # ---- lazy rebuild ----------------------------------------------------- #
    def _ensure(self):
        if self._built is not None:
            return
        import torch
        from transformers import (AutoModelForSequenceClassification,
                                   AutoTokenizer)

        self._device = "cuda" if torch.cuda.is_available() else "cpu"
        built = []
        for m in self._members:
            # tokenizer: dump bytes to a temp dir, then load with member kwargs
            tokdir = tempfile.mkdtemp(prefix="ens_tok_")
            for name, data in m["tokenizer_files"].items():
                with open(os.path.join(tokdir, name), "wb") as fh:
                    fh.write(data)
            tok = AutoTokenizer.from_pretrained(tokdir, **m.get("tokenizer_kwargs", {}))

            # model: rebuild from config + state_dict (no hub download)
            model = AutoModelForSequenceClassification.from_config(m["config"])
            model.load_state_dict(m["state_dict"])
            model.to(self._device)
            model.eval()

            built.append({"tok": tok, "model": model,
                          "is_bertweet": bool(m.get("is_bertweet", False))})
        self._built = built

    # ---- inference -------------------------------------------------------- #
    @staticmethod
    def _log_softmax(z):
        import numpy as np
        z = np.asarray(z, dtype=np.float64)
        z = z - z.max(axis=1, keepdims=True)
        return z - np.log(np.exp(z).sum(axis=1, keepdims=True))

    def _member_logits(self, member, texts):
        """Forward pass for one member -> raw logits array (n, 3)."""
        import numpy as np
        import torch
        prepped = _prep_for_member(texts, member["is_bertweet"])
        tok, model = member["tok"], member["model"]
        chunks = []
        for i in range(0, len(prepped), self.batch_size):
            batch = prepped[i:i + self.batch_size]
            enc = tok(batch, max_length=self.max_length, truncation=True,
                      padding=True, return_tensors="pt")
            enc = {k: v.to(self._device) for k, v in enc.items()}
            with torch.no_grad():
                logits = model(**enc).logits
            chunks.append(logits.detach().cpu().numpy())
        return np.vstack(chunks).astype(np.float64)

    def predict(self, X: Sequence[str]):
        """Return a list of API labels in {-1, 0, 1} for the input texts."""
        import numpy as np
        if isinstance(X, str):
            X = [X]
        texts = [normalize_text(t) for t in X]
        if len(texts) == 0:
            return []
        self._ensure()

        # mean of per-member log-softmax (= log geometric mean of probabilities)
        lp_sum = None
        for member in self._built:
            lp = self._log_softmax(self._member_logits(member, texts))
            lp_sum = lp if lp_sum is None else lp_sum + lp
        ens = lp_sum / len(self._built)

        ens = ens + np.asarray(self.offsets, dtype=np.float64)   # threshold tuning
        idx = ens.argmax(axis=1)
        return [int(self.index_to_api[int(j)]) for j in idx]

    # convenience: averaged ensemble probabilities (post-offset, softmaxed)
    def predict_proba(self, X: Sequence[str]):
        import numpy as np
        if isinstance(X, str):
            X = [X]
        texts = [normalize_text(t) for t in X]
        self._ensure()
        lp_sum = None
        for member in self._built:
            lp = self._log_softmax(self._member_logits(member, texts))
            lp_sum = lp if lp_sum is None else lp_sum + lp
        ens = lp_sum / len(self._built) + np.asarray(self.offsets, dtype=np.float64)
        ens = ens - ens.max(axis=1, keepdims=True)
        p = np.exp(ens)
        return p / p.sum(axis=1, keepdims=True)


Overwriting sentiment_deploy_ensemble.py


In [3]:
import importlib, sentiment_deploy_ensemble
importlib.reload(sentiment_deploy_ensemble)
from sentiment_deploy_ensemble import (normalize_text, cardiff_preprocess,
                                       EnsembleVectorizer, EnsembleClassifier)

import pandas as pd, numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report

def _find_col(cols, candidates):
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    return None

def normalize_labels_to_3(series):
    s = pd.to_numeric(series, errors="coerce")
    valid_s = s.dropna()
    if valid_s.empty:
        return s
    u = set(int(v) for v in valid_s.unique())
    if u <= {-1, 0, 1}:   return s.map({-1: 0, 0: 1, 1: 2})
    if u <= {0, 1, 2}:    return s
    if u <= {1,2,3,4,5}:  return s.map({1:0, 2:0, 3:1, 4:2, 5:2})
    raise ValueError(f"Unrecognised label set: {u}")

def class_weights(y):
    w = compute_class_weight("balanced", classes=np.array([0,1,2]), y=np.asarray(y))
    return w.astype("float32")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1_macro": f1_score(labels, preds, average="macro"),
            "accuracy": accuracy_score(labels, preds)}
print("Helpers ready.")

Helpers ready.


## 2. Load the canonical split (same split as Routes A/B and the 0.7580 run)

In [4]:
raw = pd.read_csv(SPLIT_CSV)
print("Loaded", SPLIT_CSV, "shape", raw.shape, "| columns:", list(raw.columns))

text_col  = TEXT_COL  or _find_col(raw.columns, ["text","review_cleaned","review","document","content","sentence"])
label_col = LABEL_COL or _find_col(raw.columns, ["sentiment_encoded","label","sentiment","target","y","numericvalue"])
split_col = SPLIT_COL or _find_col(raw.columns, ["split","set","partition","subset","fold"])
assert text_col and label_col, f"Could not find text/label columns in {list(raw.columns)}"
print(f"Using text='{text_col}', label='{label_col}', split='{split_col}'")

raw["_text"]  = raw[text_col].astype(str).map(normalize_text)
raw["labels"] = normalize_labels_to_3(raw[label_col])
raw = raw.dropna(subset=["labels"]).reset_index(drop=True)
raw["labels"] = raw["labels"].astype("int64")

if split_col is not None:
    s = raw[split_col].astype(str).str.lower().str.strip()
    train_df = raw[s.isin(TRAIN_KEYS)].copy()
    val_df   = raw[s.isin(VAL_KEYS)].copy()
    test_df  = raw[s.isin(TEST_KEYS)].copy()
    if len(val_df) == 0:
        from sklearn.model_selection import train_test_split
        train_df, val_df = train_test_split(train_df, test_size=0.1,
            stratify=train_df["labels"], random_state=SEED)
        print("No val partition found -> carved 10% val out of train.")
else:
    raise RuntimeError("No split column found. The split MUST match the 0.7580 run "
                       "or the number is meaningless. Fix data_split.csv.")

val_true  = np.array(val_df["labels"].tolist())
test_true = np.array(test_df["labels"].tolist())
for name, d in [("train",train_df),("val",val_df),("test",test_df)]:
    dist = d["labels"].value_counts().sort_index().to_dict()
    print(f"{name:5s}: {len(d):>7d}  dist(0/1/2)={dist}")

Loaded /kaggle/input/datasets/tanhvnh/sentiment-routec/data_split.csv shape (255083, 6) | columns: ['Unnamed: 0', 'Text', 'clean_eda', 'sentiment', 'Type', 'split']
Using text='Text', label='sentiment', split='split'
train:  178557  dist(0/1/2)={0: 62851, 1: 44461, 2: 71245}
val  :   38263  dist(0/1/2)={0: 13468, 1: 9528, 2: 15267}
test :   38263  dist(0/1/2)={0: 13468, 1: 9528, 2: 15267}


## 3. Tokeniser + dataset helpers (verbatim from the 0.7580 notebook)

In [5]:
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch
from torch.utils.data import Dataset

def make_tokenizer(model_name):
    if "bertweet" in model_name.lower():
        return AutoTokenizer.from_pretrained(model_name, normalization=True, use_fast=False)
    return AutoTokenizer.from_pretrained(model_name)

def tokenizer_kwargs_for(model_name):
    if "bertweet" in model_name.lower():
        return {"normalization": True, "use_fast": False}
    return {}

def _prep_texts(texts, model_name):
    if "bertweet" in model_name.lower():
        return list(texts)
    return [cardiff_preprocess(t) for t in texts]

class TokenizedDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len, chunk=1000):
        ids, masks = [], []
        texts = list(texts)
        for i in range(0, len(texts), chunk):
            b = tokenizer(texts[i:i+chunk], truncation=True, max_length=max_len)
            ids.extend(b["input_ids"]); masks.extend(b["attention_mask"])
        self.input_ids = ids; self.attention_mask = masks
        self.labels = [int(x) for x in labels]
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        if isinstance(idx, str):
            return getattr(self, "labels" if idx in ("label","labels") else idx)
        return {"input_ids": self.input_ids[idx],
                "attention_mask": self.attention_mask[idx],
                "labels": self.labels[idx]}

def encode(df, tok, model_name):
    texts = _prep_texts(df["_text"].tolist(), model_name)
    return TokenizedDataset(texts, df["labels"].tolist(), tok, MAX_LEN)
print("Tokeniser helpers ready.")

Tokeniser helpers ready.


## 4. Weighted trainer (only used when a member must be retrained)

In [6]:
import inspect, torch.nn as nn
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback)
_TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._cw = (torch.tensor(class_weight, dtype=torch.float)
                    if class_weight is not None else None)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self._cw.to(logits.device) if self._cw is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def make_args(output_dir, lr, epochs, do_eval=True):
    common = dict(output_dir=str(output_dir), learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO, seed=SEED,
        fp16=FP16, group_by_length=GROUP_BY_LENGTH, logging_steps=100,
        report_to="none", save_total_limit=1)
    eval_key = ("eval_strategy" if "eval_strategy" in _TA_PARAMS else
                "evaluation_strategy" if "evaluation_strategy" in _TA_PARAMS else None)
    if eval_key:
        common[eval_key] = "epoch" if do_eval else "no"
    if do_eval and EARLY_STOP and eval_key:
        common.update(save_strategy="epoch", load_best_model_at_end=True,
                      metric_for_best_model="f1_macro", greater_is_better=True)
    common = {k: v for k, v in common.items() if k in _TA_PARAMS}
    return TrainingArguments(**common)

def new_model(model_name):
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS)

def train_member(model_name, tok):
    """Fine-tune one member on full train with the WINNING config; return live model."""
    cw = class_weights(train_df["labels"]) if BEST_WEIGHTED else None
    short = model_name.split("/")[-1]
    args = make_args(OUT_DIR / f"final_{short}", BEST_LR, EPOCHS, do_eval=True)
    trainer = WeightedTrainer(
        model=new_model(model_name), args=args, class_weight=cw,
        train_dataset=encode(train_df, tok, model_name),
        eval_dataset=encode(val_df, tok, model_name),
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(EARLY_STOP)] if EARLY_STOP else None)
    trainer.train()
    return trainer.model           # best model (load_best_model_at_end=True)
print("Trainer ready.")

Trainer ready.


## 5. Load-or-train both members, collect val/test logits

For each model: look for the best checkpoint your original run saved under
`route_c_out/final_<short>/checkpoint-*`. **Found → load it (no retrain, the
guaranteed-equivalent path). Missing → retrain with the fixed config.** Either
way we keep the live `(model, tokenizer)` so we can capture their weights into
the pickle, and we run a plain batched forward pass to get logits.

In [7]:
import glob, os, gc

def find_checkpoint(model_name):
    short = model_name.split("/")[-1]
    base = OUT_DIR / f"final_{short}"
    cks = sorted(glob.glob(str(base / "checkpoint-*")),
                 key=lambda p: int(p.split("-")[-1]))
    # prefer an explicit best, else the highest-step checkpoint, else the dir itself
    if cks:
        return cks[-1]
    if (base / "config.json").exists():
        return str(base)
    return None

@torch.no_grad()
def infer_logits(model, tok, df, model_name, bs=EVAL_BS):
    """Plain batched forward pass -> raw logits (n, 3). Matches serve-time path."""
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    texts = _prep_texts(df["_text"].tolist(), model_name)
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], max_length=MAX_LEN, truncation=True,
                  padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        out.append(model(**enc).logits.detach().cpu().numpy())
    return np.vstack(out).astype(np.float64)

live_members = []                  # (model_name, model, tokenizer) for packaging
val_logits_per_model, test_logits_per_model, single_f1 = [], [], {}

for model_name in MODELS:
    short = model_name.split("/")[-1]
    print("="*72); print("Member:", model_name); print("="*72)
    tok = make_tokenizer(model_name)
    ckpt = find_checkpoint(model_name)
    if ckpt:
        print(f"  -> loading existing checkpoint: {ckpt}  (NO retrain)")
        model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=NUM_LABELS)
    else:
        print(f"  -> no checkpoint found under route_c_out/final_{short} -> RETRAINING "
              f"(lr={BEST_LR}, weighted={BEST_WEIGHTED}, epochs={EPOCHS})")
        model = train_member(model_name, tok)

    v = infer_logits(model, tok, val_df,  model_name)
    t = infer_logits(model, tok, test_df, model_name)
    val_logits_per_model.append(v); test_logits_per_model.append(t)
    f1_s = f1_score(test_true, np.argmax(t, 1), average="macro")
    single_f1[model_name] = f1_s
    print(f"  [{short}] single-model TEST macro-F1 (argmax) = {f1_s:.4f}")

    live_members.append((model_name, model, tok))
    gc.collect()
    try: torch.cuda.empty_cache()
    except Exception: pass

print("\nCollected logits for", len(MODELS), "members.")

Member: vinai/bertweet-base


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  -> loading existing checkpoint: route_c_out/final_twitter-roberta-base-sentiment-latest/checkpoint-11160  (NO retrain)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  [twitter-roberta-base-sentiment-latest] single-model TEST macro-F1 (argmax) = 0.7593

Collected logits for 2 members.


## 6. Log-average ensemble + per-class threshold tuning → reproduce 0.7580

In [8]:
from sklearn.metrics import f1_score, classification_report

def log_softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))

def tune_thresholds(val_scores, val_labels, grid=None, passes=4):
    if grid is None:
        grid = np.linspace(-3.0, 3.0, 61)
    n_cls = val_scores.shape[1]; offs = np.zeros(n_cls)
    def f1_at(o): return f1_score(val_labels, np.argmax(val_scores + o, 1), average="macro")
    best = f1_at(offs)
    for _ in range(passes):
        improved = False
        for c in range(n_cls):
            best_g = offs[c]
            for g in grid:
                trial = offs.copy(); trial[c] = g
                s = f1_at(trial)
                if s > best + 1e-9:
                    best, best_g, improved = s, g, True
            offs[c] = best_g
        if not improved: break
    return offs, best

LABELS = ["Negative(0)", "Neutral(1)", "Positive(2)"]
def report(true, pred, header):
    f1 = f1_score(true, pred, average="macro")
    print(f"\n=== {header}: macro-F1 = {f1:.4f} ({f1*100:.1f}%) ===")
    print(classification_report(true, pred, target_names=LABELS, digits=3))
    return f1

val_ens  = np.mean([log_softmax_np(v) for v in val_logits_per_model],  axis=0)
test_ens = np.mean([log_softmax_np(t) for t in test_logits_per_model], axis=0)

f1_argmax = report(test_true, np.argmax(test_ens, 1), "ENSEMBLE (log-avg) + argmax")

if FORCE_OFFSETS is not None:
    offsets = np.asarray(FORCE_OFFSETS, dtype=np.float64)
    val_best = f1_score(val_true, np.argmax(val_ens + offsets, 1), average="macro")
    print(f"\nUsing FORCED offsets: {offsets}  | val macro-F1 = {val_best:.4f}")
else:
    offsets, val_best = tune_thresholds(val_ens, val_true, grid=THRESH_GRID)
    print(f"\nTuned per-class offsets (fit on val): {np.round(offsets,3)}"
          f"  | val macro-F1 after tuning = {val_best:.4f}")

test_pred = np.argmax(test_ens + offsets, 1)
f1_tuned  = report(test_true, test_pred,
                   "ENSEMBLE (log-avg) + per-class threshold tuning  [FINAL]")

print("\n" + "-"*56); print("SUMMARY (TEST macro-F1)"); print("-"*56)
for k, v in single_f1.items():
    print(f"  {k.split('/')[-1]:<38} {v:.4f}")
print(f"  {'ensemble (argmax)':<38} {f1_argmax:.4f}")
print(f"  {'ensemble + thresholds  [FINAL]':<38} {f1_tuned:.4f}  (Δ {f1_tuned-f1_argmax:+.4f})")


=== ENSEMBLE (log-avg) + argmax: macro-F1 = 0.7621 (76.2%) ===
              precision    recall  f1-score   support

 Negative(0)      0.814     0.823     0.819     13468
  Neutral(1)      0.568     0.658     0.610      9528
 Positive(2)      0.911     0.811     0.858     15267

    accuracy                          0.777     38263
   macro avg      0.764     0.764     0.762     38263
weighted avg      0.791     0.777     0.782     38263


Tuned per-class offsets (fit on val): [0.1 0.  0.2]  | val macro-F1 after tuning = 0.7643

=== ENSEMBLE (log-avg) + per-class threshold tuning  [FINAL]: macro-F1 = 0.7624 (76.2%) ===
              precision    recall  f1-score   support

 Negative(0)      0.806     0.835     0.821     13468
  Neutral(1)      0.581     0.632     0.605      9528
 Positive(2)      0.902     0.824     0.862     15267

    accuracy                          0.780     38263
   macro avg      0.763     0.764     0.762     38263
weighted avg      0.788     0.780     0.783  

## 7. Package → `route_c_ensemble.model`

Capture each live member's config + weights + tokenizer files into the picklable
`EnsembleClassifier`, bake in the offsets, and write the `{"vectorizer","classifier"}`
dict the API expects. Nothing here depends on `route_c_out/` after this cell — the
weights live inside the pickle.

In [14]:
import pickle, tempfile

MODEL_FILE="/kaggle/working/route_c_out/route_c_ensemble.model"

def capture_member(model_name, model, tokenizer):
    cfg = model.config
    state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    with tempfile.TemporaryDirectory() as d:
        tokenizer.save_pretrained(d)
        files = {}
        for name in os.listdir(d):
            p = os.path.join(d, name)
            if os.path.isfile(p):
                with open(p, "rb") as fh:
                    files[name] = fh.read()
    return {"name": model_name,
            "is_bertweet": "bertweet" in model_name.lower(),
            "tokenizer_kwargs": tokenizer_kwargs_for(model_name),
            "config": cfg, "state_dict": state, "tokenizer_files": files}

members_payload = [capture_member(n, m, t) for (n, m, t) in live_members]

clf = EnsembleClassifier(members=members_payload,
                         offsets=offsets.tolist(),
                         max_length=MAX_LEN, batch_size=INFER_BS)
deployment = {"vectorizer": EnsembleVectorizer(), "classifier": clf}

with open(MODEL_FILE, "wb") as f:
    pickle.dump(deployment, f)
print(f"Wrote {MODEL_FILE}  ({os.path.getsize(MODEL_FILE)/1e6:.1f} MB)")
print("Offsets baked in:", offsets.tolist())


from IPython.display import FileLink
FileLink('route_c_ensemble.model')

Wrote /kaggle/working/route_c_out/route_c_ensemble.model  (1043.4 MB)
Offsets baked in: [0.10000000000000009, 0.0, 0.20000000000000018]


/kaggle/working/route_c_ensemble.model